In [ ]:
from pathlib import Path
from typing import Optional

import pandas as pd

from scripts.preprocess_wsi_tiles import (
    DATA_PATH,
    PROJECT_DATA_PATH,
    IMAGE_DST_PATH,
    CPTAC_RAW_PATH,
    run_tcga_tiling,
    run_cptac_tiling,
    set_seed,
)

set_seed(42)

TARGET_MPP = 1.0
TILE_SIZE = 1024
TISSUE_THRESHOLD = 0.15

# Low-res tissue mask에서 먼저 흰 배경 tile을 제외한 뒤 실제 tile을 읽습니다.
PREFILTER_TISSUE_THRESHOLD = 0.01
TCGA_TISSUE_MASK_DOWNSAMPLE = 64

# 처음 테스트할 때는 1 또는 3으로 설정하고, 확인 후 None으로 바꾸세요.
DEBUG_MAX_CASES: Optional[int] = 1

# case 단위 병렬 처리입니다. CPTAC는 DICOM frame decoding이 무거워 너무 크게 잡지 않습니다.
TCGA_MAX_WORKERS = 4
CPTAC_MAX_WORKERS = 2
SHOW_TILE_PROGRESS = False  # True는 max_workers=1 디버깅 때 가장 보기 좋습니다.

SKIP_EXISTING = True
OVERWRITE_METADATA = False

# PNG는 무손실이지만 느리고 용량이 큽니다. feature extraction 목적이면 jpg도 고려 가능합니다.
IMAGE_FORMAT = "png"  # "png" or "jpg"
JPEG_QUALITY = 90

TCGA_COHORT_PATH = Path("outputs/data_verification/tcga_survival_cohort_candidate.csv")
CPTAC_COHORT_PATH = Path("outputs/data_verification/cptac_survival_cohort_candidate.csv")
CPTAC_WSI_SERIES_PATH = CPTAC_RAW_PATH / "matched" / "cptac_pda_matched_wsi_series.csv"

print("PROJECT_DATA_PATH:", PROJECT_DATA_PATH.resolve())
print("IMAGE_DST_PATH:", IMAGE_DST_PATH.resolve())
print("TARGET_MPP:", TARGET_MPP, "TILE_SIZE:", TILE_SIZE)
print("DEBUG_MAX_CASES:", DEBUG_MAX_CASES)


In [ ]:
# TCGA-PAAD SVS WSI tiling: mpp 1.0, 1024 x 1024
# 개선점: low-res tissue mask로 흰 배경 tile 좌표를 먼저 제외하고, case 단위 병렬 처리합니다.

tcga_summary_df = run_tcga_tiling(
    cohort_path=TCGA_COHORT_PATH,
    max_cases=DEBUG_MAX_CASES,
    max_workers=TCGA_MAX_WORKERS,
    target_mpp=TARGET_MPP,
    tile_size=TILE_SIZE,
    tissue_threshold=TISSUE_THRESHOLD,
    prefilter_tissue_threshold=PREFILTER_TISSUE_THRESHOLD,
    tissue_mask_downsample=TCGA_TISSUE_MASK_DOWNSAMPLE,
    skip_existing=SKIP_EXISTING,
    overwrite_metadata=OVERWRITE_METADATA,
    image_format=IMAGE_FORMAT,
    jpeg_quality=JPEG_QUALITY,
    show_tile_progress=SHOW_TILE_PROGRESS,
)

display(tcga_summary_df)
print("saved:", IMAGE_DST_PATH / "TCGA_PAAD" / "tile_summary.csv")


In [ ]:
# CPTAC-PDAC DICOM WSI tiling: mpp 1.0, 1024 x 1024
# 개선점: DICOM 전체 pixel_array를 메모리에 올리지 않고 필요한 frame만 부분 디코딩하며, case 단위 병렬 처리합니다.

cptac_summary_df = run_cptac_tiling(
    cohort_path=CPTAC_COHORT_PATH,
    wsi_series_path=CPTAC_WSI_SERIES_PATH,
    max_cases=DEBUG_MAX_CASES,
    max_workers=CPTAC_MAX_WORKERS,
    target_mpp=TARGET_MPP,
    tile_size=TILE_SIZE,
    tissue_threshold=TISSUE_THRESHOLD,
    prefilter_tissue_threshold=PREFILTER_TISSUE_THRESHOLD,
    skip_existing=SKIP_EXISTING,
    overwrite_metadata=OVERWRITE_METADATA,
    image_format=IMAGE_FORMAT,
    jpeg_quality=JPEG_QUALITY,
    show_tile_progress=SHOW_TILE_PROGRESS,
)

display(cptac_summary_df)
print("saved:", IMAGE_DST_PATH / "CPTAC_PDAC" / "tile_summary.csv")
